# Stage 2 Notebook 52 - Exp2WW Anchor + topk-fixed K=8 matching + VFL

**The matcher fix the cls collapse really needed.** Across NB39-46 (focal/ASL/QFL/VFL on the anchor head) the cls stayed at uniform sigmoid (gap <= 0.015) regardless of loss formulation. NB44 (Hungarian 1-to-1 on anchor) crashed geometry to matched_iou=0.38. NB47 (Hungarian on K=64 query head) FINALLY broke the cls collapse (gap=0.099) -- but had weak geometry (matched_iou=0.27).

Diagnosis: dynamic-k matching's K is **estimated** from sum(top-K IoUs), so it varies batch to batch. Same prior gets K=2 in one batch, K=4 in another, sometimes 0. Per-prior labels FLICKER and cls converges to uniform sigmoid as the only stable equilibrium under noisy labels.

Hungarian fixes the flickering but with K=1 per GT, only ~5 priors per image get positive geometry supervision -- starvation that crashed NB44's matched_iou.

**Exp2WW: fix-K-by-cost matching.** Each GT lane takes the top K=8 priors by cost, with conflict resolution (each prior matches at most one GT). Properties:
- K=8 priors per GT * 5 GT = ~40 positives per image (anchor-style dense geometry supervision).
- Each prior's label is deterministic given the cost matrix -- no IoU-sum random rounding.
- Combined with VFL on continuous LineIoU regression target.

Code: new `_topk_fixed_match` in losses.py + `lane_assigner='topk_fixed'` + `topk_fixed_per_gt=8` config. Backwards compatible with all prior runs.

### Run mode
1. `DEBUG_MODE = True` smoke first (new matcher path).
2. `DEBUG_MODE = False` for 20-epoch short run at limit=3000.
3. ~30 min wall-clock with AMP.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp47_rmt_gca_anchor_topk_fixed_vfl_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp47_rmt_gca_anchor_topk_fixed_vfl_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp47_rmt_gca_anchor_topk_fixed_vfl_joint_smoke.log
OK exp47_rmt_gca_anchor_topk_fixed_vfl_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.2766 det_loss=2.9009 grad_cos=0.2241 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4983346164226532, 'gate/lane_mean': 0.5000624060630798, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp47_rmt_gca_anchor_topk_fixed_vfl_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: 3000
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp47_rmt_gca_anchor_topk_fixed_vfl_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp47_rmt_gca_anchor_topk_fixed_vfl_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp47_rmt_gca_anchor_topk_fixed_vfl_joint_short20.tar --epochs 20 --batch-size 8 --limit-val 1000 --force-extract --print-every 50 --limit-train 3000
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp47_rmt_gca_anchor_topk_fixed_vfl_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp47_rmt_gca_anchor_topk_fixed_vfl_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp47_rmt_gca_anchor_topk_fixed_vfl_joint.yaml --curve-t

0

## What to watch in Exp2WW training

Reference NB48 (anchor + dynamic-k + VFL + 70K data): matched_iou=0.544, decoded_f1=0.05, gap=0.015.
Reference NB47 (K=64 query + Hungarian + VFL): val_lane_f1=0.246, gap=0.099, matched_iou=0.27.

Pass criteria at epoch 20 (limit=3000):
- **`pos_score - neg_score >= 0.05`** -- stable labels should give cls room to discriminate.
- **`val/matched_line_iou >= 0.45`** -- preserve anchor-head geometry (denser supervision than NB44 Hungarian's 0.38).
- **`val/lane/decoded_f1 >= 0.10`** -- 2x NB48; cls finally ranks correctly.
- **`val/lane_f1 >= 0.10` and `val/lane_best_f1 >= 0.20`** -- legacy F1 metrics show real discrimination (vs anchor head's 0.0 across NB39-46).

Failure signals:
- gap < 0.02: stable labels alone aren't enough; per-prior features fundamentally non-discriminative. Move to query-based heads (Exp2XX).
- matched_iou < 0.35: K=8 conflict resolution starves geometry; lower to K=6.
- gap >= 0.05 AND decoded_f1 >= 0.10: matcher was the bug. Combine with NB48's full data in Exp2YY-style follow-up.